In [ ]:
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as v2
import os
import cv2
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
import scipy.io as io
import numpy as np
! wget https://zenodo.org/record/4126613/files/CALTECH.zip -O caltech.zip && unzip -q caltech.zip

class Caltech(Dataset):
    def __init__(self, root_img, root_annot, transform=None):
        self.root_img = root_img
        self.root_annot = root_annot
        self.transform = transform

        self.imgs_path = []
        self.annots_path = []
        self.labels = []

        for class_idx, class_name in enumerate(os.listdir(root_img)):
            class_path = os.path.join(root_img, class_name)
            class_annot_path = os.path.join(root_annot, class_name)
            for img_name in os.listdir(class_path):
                annot_name = img_name.replace(".jpg", ".mat").replace("image", "annotation")
                img_path = os.path.join(class_path, img_name)
                annot_path = os.path.join(class_annot_path, annot_name)
                self.imgs_path.append(img_path)
                self.annots_path.append(annot_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.imgs_path)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs_path[idx])
        mat = io.loadmat(self.annots_path[idx])
        label = self.labels[idx]

        h, w, c = img.shape
        y_min, y_max, x_min, x_max = mat['box_coord'][0]

        annot = torch.tensor([x_min / w, y_min / h, x_max / w, y_max / h]).float()

        if self.transform:
            img = self.transform(img)

        return img, annot, label


class localizer(nn.Module):
    def __init__(self):
        super(localizer, self).__init__()
        self.feat_ext = torchvision.models.resnet18(weights=None)
        in_features = self.feat_ext.fc.in_features
        # self.model.fc = nn.Linear(in_features=self.model.fc.in_features, out_features=4)
        self.feat_ext.fc = nn.Identity()

        self.loc_head = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=4),
            nn.Sigmoid()
        )
        self.cls_head = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=3),
        )

    def forward(self, x):
        x = self.feat_ext(x)
        bbox = self.loc_head(x)
        pred = self.cls_head(x)

        return pred, bbox


def train(loader, model, cls_cri, reg_cri, optimizer, device):
    model.train()
    metrics = {
        "reg_loss": [],
        "cls_loss": [],
        "loss": [],
        "acc": [],
    }
    for data, annot, label in loader:
        optimizer.zero_grad()

        data = data.to(device)
        label = label.to(device)
        annot = annot.to(device)

        pred, bboxes = model(data)
        cls_loss = cls_cri(pred, label)
        reg_loss = reg_cri(bboxes, annot)
        loss = cls_loss + reg_loss
        loss.backward()
        optimizer.step()

        acc = accuracy_score(label.detach().cpu(), pred.argmax(dim=1).detach().cpu())
        metrics['acc'].append(acc)
        metrics['loss'].append(loss.item())
        metrics['reg_loss'].append(reg_loss.item())
        metrics['cls_loss'].append(cls_loss.item())

    return metrics


def val(loader, model, cls_cri, reg_cri, device):
    model.eval()
    metrics = {
        "reg_loss": [],
        "cls_loss": [],
        "loss": [],
        "acc": [],
    }
    with torch.no_grad():
        for data, annot, label in loader:
            data = data.to(device)
            label = label.to(device)
            annot = annot.to(device)

            pred, bboxes = model(data)
            cls_loss = cls_cri(pred, label)
            reg_loss = reg_cri(bboxes, annot)
            loss = cls_loss + reg_loss

            acc = accuracy_score(label.detach().cpu(), pred.argmax(dim=1).detach().cpu())
            metrics['acc'].append(acc)
            metrics['loss'].append(loss.item())
            metrics['reg_loss'].append(reg_loss.item())
            metrics['cls_loss'].append(cls_loss.item())

    return metrics


root_img = "/content/CALTECH/CALTECH_Dataset/"
root_annot = "/content/CALTECH/CALTECH_Annotations/"

transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize((224, 224)),
    v2.ToTensor()
])

full_dataset = Caltech(root_img, root_annot, transform)

# Define split sizes
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = localizer()
cls_criterion = nn.CrossEntropyLoss()
reg_criterion = nn.MSELoss()

model = model.to(device)
cls_criterion = cls_criterion.to(device)
reg_criterion = reg_criterion.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
epochs = 50
for epoch in range(epochs):
    train_metrics = train(loader, model, cls_criterion, reg_criterion, optimizer, device)
    val_metrics = val(loader, model, cls_criterion, reg_criterion, device)
    print(
        f"Epoch {epoch + 1}/{epochs}: Train loss: {np.mean(train_metrics['loss'])}, Train acc: {np.mean(train_metrics['acc'])}, cls loss: {np.mean(train_metrics['cls_loss'])}, reg loss: {np.mean(train_metrics['reg_loss'])}")
    print(
        f"Epoch {epoch + 1}/{epochs}: Val loss: {np.mean(val_metrics['loss'])}, Val acc: {np.mean(val_metrics['acc'])}, Val cls loss: {np.mean(val_metrics['cls_loss'])}, Val reg loss: {np.mean(val_metrics['reg_loss'])}")

model = model.to('cpu')
for img, annot, label in val_loader:
    pred, bboxes = model(img)
    break
idx = 13
x_min, y_min, x_max, y_max = annot[idx]
img_ = cv2.cvtColor(img[idx].permute(1, 2, 0).numpy(), cv2.COLOR_BGR2RGB)
img_ = cv2.rectangle(img_, (int(x_min * 224), int(y_min * 224)), (int(x_max * 224), int(y_max * 224)), (0, 255, 0), 2)
img_ = cv2.circle(img_, (int(x_min * 224), int(y_min * 224)), radius=0, color=(255, 0, 255), thickness=10)
img_ = cv2.circle(img_, (int(x_max * 224), int(y_max * 224)), radius=0, color=(255, 0, 255), thickness=10)
x_min, y_min, x_max, y_max = bboxes[idx]
img_ = cv2.rectangle(img_, (int(x_min * 224), int(y_min * 224)), (int(x_max * 224), int(y_max * 224)), (255, 0, 0), 2)
img_ = cv2.circle(img_, (int(x_min * 224), int(y_min * 224)), radius=0, color=(0, 255, 255), thickness=10)
img_ = cv2.circle(img_, (int(x_max * 224), int(y_max * 224)), radius=0, color=(0, 255, 255), thickness=10)
plt.imshow(img_)
# 2025-06-01

from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as v2
import os
import cv2
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
import scipy.io as io
import numpy as np
! wget https://zenodo.org/record/4126613/files/CALTECH.zip -O caltech.zip && unzip -q caltech.zip
img = cv2.imread("/content/CALTECH/CALTECH_Dataset/elephant/image_0001.jpg")
mat = io.loadmat("/content/CALTECH/CALTECH_Annotations/elephant/annotation_0001.mat")
h, w, c = img.shape
y_min, y_max, x_min, x_max = mat['box_coord'][0]
annot = [x_min, y_min, x_max, y_max]
img
transform = v2.Compose([
    v2.ToPILImage(),
    v2.RandomHorizontalFlip(p=1.),
    v2.RandomRotation(degrees=(-10, 10)),
])
img_t = transform(img)
img_t
# train_trainsform = torchvision.transforms.Compose([
#                 torchvision.transforms.ToPILImage(),
#                 torchvision.transforms.Resize(256),
#                 torchvision.transforms.CenterCrop(224),
#                 torchvision.transforms.RandomHorizontalFlip(p=0.3),
#                 v2.RandomPerspective(distortion_scale=0.1, p=0.3),
#                 v2.RandomChoice([
#                     v2.ColorJitter(brightness=.3, hue=.2),
#                     v2.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)),
#                     v2.RandomRotation(degrees=(-20, 20)),
#                 ], p=[0.3, 0.3, 0.3]),
# ])
# train_trainsform(img)
x_min, y_min, x_max, y_max = annot
img_ = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_ = cv2.rectangle(img_, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
plt.imshow(img_)

x_min, y_min, x_max, y_max = annot
img_ = cv2.cvtColor(np.array(img_t), cv2.COLOR_BGR2RGB)
img_ = cv2.rectangle(img_, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
plt.imshow(img_)
import albumentations as A

transform = A.Compose([
    A.HorizontalFlip(p=1.),
], bbox_params=A.BboxParams(format='pascal_voc'))
output = transform(image=img, bboxes=[annot])
output.keys()
output['image']
output['bboxes'][0], annot
x_min, y_min, x_max, y_max = annot
img_ = cv2.cvtColor(np.array(output['image']), cv2.COLOR_BGR2RGB)
img_ = cv2.rectangle(img_, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
x_min, y_min, x_max, y_max = output['bboxes'][0]
img_ = cv2.rectangle(img_, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 0, 255), 2)
plt.imshow(img_)
transform = A.Compose([
    A.Resize(height=224, width=224),
    # A.HorizontalFlip(p=1.),
    A.RandomRotate90(p=1.),
    # A.HorizontalFlip(p=1.),
    # A.RandomRotate90(p=1.),
], bbox_params=A.BboxParams(format='pascal_voc'))

output = transform(image=img, bboxes=[annot])
x_min, y_min, x_max, y_max = annot
img_ = cv2.cvtColor(np.array(output['image']), cv2.COLOR_BGR2RGB)
img_ = cv2.rectangle(img_, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
x_min, y_min, x_max, y_max = output['bboxes'][0]
img_ = cv2.rectangle(img_, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 0, 255), 2)
plt.imshow(img_)
output['bboxes']
output['bboxes'][0]
from copy import copy


class Caltech(Dataset):
    def __init__(self, root_img, root_annot, torch_transform=None, a_transfrom=None):
        self.root_img = root_img
        self.root_annot = root_annot
        self.torch_transform = torch_transform
        self.a_transform = a_transform

        self.imgs_path = []
        self.annots_path = []
        self.labels = []

        for class_idx, class_name in enumerate(os.listdir(root_img)):
            class_path = os.path.join(root_img, class_name)
            class_annot_path = os.path.join(root_annot, class_name)
            for img_name in os.listdir(class_path):
                annot_name = img_name.replace(".jpg", ".mat").replace("image", "annotation")
                img_path = os.path.join(class_path, img_name)
                annot_path = os.path.join(class_annot_path, annot_name)
                self.imgs_path.append(img_path)
                self.annots_path.append(annot_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.imgs_path)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs_path[idx])
        mat = io.loadmat(self.annots_path[idx])
        label = self.labels[idx]
        img_o = copy(img)
        y_min, y_max, x_min, x_max = mat['box_coord'][0]
        annot = [x_min, y_min, x_max, y_max]
        output = self.a_transform(image=img, bboxes=[annot])
        img = self.torch_transform(output['image'])
        annot2 = torch.tensor(output['bboxes'][0]).float()

        return img_o, img, torch.tensor(annot).float(), annot2, label


a_transform = A.Compose([
    # A.RandomResizedCrop(size=(224, 224), scale= (1.,1.), ratio=(1.0, 1.0), p=1),
    A.Resize(224, 224),

    # A.HorizontalFlip(p=1),
    A.Rotate(limit=45, p=1),
    # A.OneOf([
    #     A.RandomBrightnessContrast(p=1),
    #     A.RandomGamma(p=1),
    #     A.GaussianBlur(p=1),
    # ], p=0.5),
], bbox_params=A.BboxParams(format='pascal_voc'))

torch_transform = v2.Compose([
    v2.ToPILImage(),
    v2.ToTensor(),
    # v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = Caltech(root_img="/content/CALTECH/CALTECH_Dataset/", root_annot="/content/CALTECH/CALTECH_Annotations/",
                  torch_transform=torch_transform, a_transfrom=a_transform)
loader = DataLoader(dataset, batch_size=1, shuffle=True)
imgs_o, imgs, bboxes, bboxes2, _ = next(iter(loader))
idx = 0
img_ = cv2.cvtColor(imgs[idx].permute(1, 2, 0).numpy(), cv2.COLOR_BGR2RGB)
x_min, y_min, x_max, y_max = bboxes[idx]
img_ = cv2.rectangle(img_, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 255, 0), 2)
x_min, y_min, x_max, y_max = bboxes2[idx]
img_ = cv2.rectangle(img_, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 0, 255), 2)
# img_ = cv2.circle(img_, (int(x_min * 224), int(y_min * 224)), radius=0, color=(0, 255, 255), thickness=10)
# img_ = cv2.circle(img_, (int(x_max * 224), int(y_max * 224)), radius=0, color=(0, 255, 255), thickness=10)
plt.imshow(img_)
img_ = cv2.cvtColor(imgs_o[idx].numpy(), cv2.COLOR_BGR2RGB)
plt.imshow(img_)
# 2025-05-11
## Object Detection
### YOLO Algorithm
pip
install
ultralytics
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.predict("street.jpg")
# Process results list
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    # result.show()  # display to screen
    # result.save(filename="result.jpg")  # save to disk
